In [ ]:
# Carl et al. (in prep.): measurement script
# Copyright (C) 2026 Friedrich Carl
# This file is part of Carl et al. (in prep.).
#
# The code is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# The code is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License
# along with MyProject.  If not, see  https://www.gnu.org/licenses/gpl-3.0.txt.

In [ ]:
"""This is the script to extract synthetic datasets from a mesh, interpolate these datasets in 3D with GemPy (de la Varga et al., 2019) and 
create the precentile threshold models, as described in Carl et al. (in prep., 2026). For explanations on why certain steps in the code are 
being executed (that way), please refer to the paper.
Please remember to change all paths/directories in this script according to your system"""

In [ ]:
"""Code to randomly extract the synthetic datasets. Data configurations have to be manually defined at the top. At the end, a simple visualization
tool is implemented to inspect a single dataset alongside the initial mesh"""
import os
import math
import random
import json
import numpy as np
import vtk
import pyvista as pv
from shapely.geometry import LineString, MultiLineString, Point, mapping
from shapely.ops import linemerge
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# === User settings ===
input_vtk_path = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Juliusburg_Koestorf-Rosenthal.vtk"
output_base_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models"
N_DATASETS = 100
SLICES_PER_DATASET = 1 # number of sections to create in the data configuration
BOREHOLES_PER_DATASET = 2 # number of boreholes to create
MAX_ATTEMPTS = 25
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

"""switch between unstructured grid and polydata if needed"""
# === Helper VTK functions ===
def read_unstructured_grid(path):
    reader = vtk.vtkUnstructuredGridReader()
   # reader = vtk.vtkPolyDataReader()
    reader.SetFileName(path)
    reader.Update()
    return reader.GetOutput()


def compute_bounds(vtk_dataset):
    return vtk_dataset.GetBounds()


def create_vertical_plane(normal_xy, origin):
    plane = vtk.vtkPlane()
    nx, ny = normal_xy
    plane.SetNormal(nx, ny, 0.0)
    plane.SetOrigin(origin[0], origin[1], origin[2])
    return plane


def cut_dataset_with_plane(vtk_dataset, plane):
    cutter = vtk.vtkCutter()
    cutter.SetCutFunction(plane)
    cutter.SetInputData(vtk_dataset)
    cutter.Update()
    poly = vtk.vtkPolyData()
    poly.ShallowCopy(cutter.GetOutput())
    return poly


def extract_surface(vtk_dataset):
    surf_filter = vtk.vtkDataSetSurfaceFilter()
    surf_filter.SetInputData(vtk_dataset)
    surf_filter.Update()
    poly = vtk.vtkPolyData()
    poly.ShallowCopy(surf_filter.GetOutput())
    return poly


def compute_normals(polydata):

    normals_filter = vtk.vtkPolyDataNormals()
    normals_filter.SetInputData(polydata)

    normals_filter.ComputeCellNormalsOn()
    normals_filter.ComputePointNormalsOff()

    normals_filter.ConsistencyOn()
    normals_filter.AutoOrientNormalsOn()
    normals_filter.SplittingOff()

    normals_filter.Update()

    return normals_filter.GetOutput()


def find_closest_cell_normal(poly_with_normals, point):
    """
    Find the cell whose closest point to `point` is nearest and return its cell normal.
    Returns (nx, ny, nz) as floats. If normals are missing, returns (0,0,0).
    """
    locator = vtk.vtkCellLocator()
    locator.SetDataSet(poly_with_normals)
    locator.BuildLocator()
    closestPoint = [0.0, 0.0, 0.0]
    cellId = vtk.reference(0)
    subId = vtk.reference(0)
    dist2 = vtk.reference(0.0)
    locator.FindClosestPoint(point, closestPoint, cellId, subId, dist2)
    normals = poly_with_normals.GetCellData().GetNormals()
    if normals and int(cellId) >= 0:
        nx, ny, nz = normals.GetTuple(int(cellId))
        return (float(nx), float(ny), float(nz))
    else:
        return (0.0, 0.0, 0.0)


def borehole_intersections(surface_polydata, x, y, zmin, zmax):

    obb = vtk.vtkOBBTree()
    obb.SetDataSet(surface_polydata)
    obb.BuildLocator()

    pts_all = []

    # upward ray
    p0 = (x, y, zmin - 1e-6)
    p1 = (x, y, zmax + 1e-6)

    pts = vtk.vtkPoints()
    ids = vtk.vtkIdList()

    obb.IntersectWithLine(p0, p1, pts, ids)

    for i in range(pts.GetNumberOfPoints()):
        pts_all.append(tuple(pts.GetPoint(i)))

    # downward ray
    pts = vtk.vtkPoints()
    ids = vtk.vtkIdList()

    obb.IntersectWithLine(p1, p0, pts, ids)

    for i in range(pts.GetNumberOfPoints()):
        pts_all.append(tuple(pts.GetPoint(i)))

    # remove duplicates
    unique = list({(round(p[0],6),round(p[1],6),round(p[2],6)):p for p in pts_all}.values())

    unique.sort(key=lambda p: p[2])

    return unique


def vtk_polydata_to_shapely_lines(poly):
    n_cells = poly.GetNumberOfCells()
    lines = []
    for cid in range(n_cells):
        cell = poly.GetCell(cid)
        npnts = cell.GetNumberOfPoints()
        if npnts < 2:
            continue
        coords = [tuple(poly.GetPoint(cell.GetPointId(j))) for j in range(npnts)]
        try:
            ls = LineString(coords)
            if ls.length > 0:
                lines.append(ls)
        except Exception:
            continue
    if not lines:
        return None
    merged = linemerge(lines)
    return merged


def shapely_to_json_feature(geom, properties=None):
    return {"type": "Feature", "geometry": mapping(geom), "properties": properties or {}}


def save_json_features(features, orientations, filename):
    fc = {
        "type": "FeatureCollection",
        "features": features,
        "orientations": orientations,
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False, indent=2)


# === Main generation ===
def generate_random_datasets(input_path, output_dir, n_datasets=10, slices_per=5, boreholes_per=5):
    vtk_data = read_unstructured_grid(input_path)
    bounds = compute_bounds(vtk_data)
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    print("Model bounds:", bounds)

    mesh_basename = os.path.splitext(os.path.basename(input_path))[0]

    # Build subfolder name dynamically, e.g. "4 sections 4 boreholes"
    combo_folder = f"{slices_per} sections {boreholes_per} boreholes"
    
    # Build full path:
    #   base dir / mesh_name / combo_folder / json
    mesh_output_dir = os.path.join(output_dir, mesh_basename, combo_folder, "json")
    
    # Create all necessary directories
    os.makedirs(mesh_output_dir, exist_ok=True)
    
    print(f"Output folder for JSON datasets:\n  {mesh_output_dir}")

    # Precompute surface & normals once for the whole mesh
    surface_poly = extract_surface(vtk_data)

    surf = pv.wrap(surface_poly)
    surf = surf.fill_holes(10000)
    
    surface_poly = surf
    surface_normals = compute_normals(surface_poly)

    for dataset_id in range(n_datasets):
        slice_geoms = []
        borehole_geoms = []            # LineStrings connecting top & bottom (kept for compatibility)
        borehole_point_geoms = []      
        orientations = []

        # --- slices ---
        slices_created = 0
        attempts = 0
        while slices_created < slices_per and attempts < slices_per * MAX_ATTEMPTS:
            attempts += 1
            theta = random.uniform(0.0, 2.0 * math.pi)
            plane_nx, plane_ny = math.cos(theta), math.sin(theta)
            ox = random.uniform(xmin, xmax)
            oy = random.uniform(ymin, ymax)
            oz = random.uniform(zmin, zmax)
            plane = create_vertical_plane((plane_nx, plane_ny), (ox, oy, oz))
            poly_slice = cut_dataset_with_plane(vtk_data, plane)
            if poly_slice.GetNumberOfPoints() == 0:
                continue
            shapely_geom = vtk_polydata_to_shapely_lines(poly_slice)
            if shapely_geom is None:
                continue
            slice_geoms.append(shapely_geom)
            slices_created += 1



            # Collect all coordinates of the section geometry
            coords = []
            if isinstance(shapely_geom, LineString):
                coords = list(shapely_geom.coords)
            elif isinstance(shapely_geom, MultiLineString):
                for part in shapely_geom.geoms:
                    coords.extend(list(part.coords))
            
            if coords:
                coords_arr = np.array(coords, dtype=float)

                
                # find index of point with maximum z
                max_z_idx = int(np.argmax(coords_arr[:, 2]))
                x_top, y_top, z_top = coords_arr[max_z_idx, 0], coords_arr[max_z_idx, 1], coords_arr[max_z_idx, 2]

                orientations.append([
                    float(x_top),
                    float(y_top),
                    float(z_top),
                    0.0, 0.0, 1.0   # upward direction at the topmost point
                ])



        # --- boreholes --- (now keep ALL intersections as point features + retain top-bottom line)
        boreholes_created = 0
        attempts = 0
        while boreholes_created < boreholes_per and attempts < boreholes_per * MAX_ATTEMPTS:
            attempts += 1
            bx = random.uniform(xmin, xmax)
            by = random.uniform(ymin, ymax)
            intersections = borehole_intersections(surface_poly, bx, by, zmin, zmax)
            if len(intersections) < 1:
                continue

            # Sort by z (ascending)
            z_sorted = sorted(intersections, key=lambda p: p[2])

            # Keep the canonical top-bottom LineString for compatibility if there are >=2 intersections
            if len(z_sorted) >= 2:
                bottom, top = z_sorted[0], z_sorted[-1]
                bls = LineString([bottom, top])
                borehole_geoms.append(bls)
            # If only one intersection, we do not create a line (but still create point features)

            # For every intersection point, create a Point feature and extract its normal -> orientation
            for p_idx, p in enumerate(z_sorted):
                # create shapely Point for this intersection (will be exported as a feature)
                borehole_point_geoms.append(Point((p[0], p[1], p[2])))
                # get cell normal at this intersection
                nx, ny, nz = find_closest_cell_normal(surface_normals, p)
                vec = np.array([nx, ny, nz], dtype=float)
                norm = np.linalg.norm(vec)
                if norm > 1e-12:
                    vec = vec / norm
                else:
                    vec = np.array([0.0, 0.0, 0.0])
                orientations.append([float(p[0]), float(p[1]), float(p[2]), float(vec[0]), float(vec[1]), float(vec[2])])

            boreholes_created += 1

        # --- write out (features: slices, borehole lines, borehole intersection points) ---
        features = []
        for i, g in enumerate(slice_geoms):
            props = {"dataset_id": dataset_id, "kind": "slice", "index": i}
            if isinstance(g, MultiLineString):
                for j, part in enumerate(g.geoms):
                    features.append(shapely_to_json_feature(part, {**props, "part": j}))
            else:
                features.append(shapely_to_json_feature(g, props))

        for i, g in enumerate(borehole_geoms):
            props = {"dataset_id": dataset_id, "kind": "borehole", "index": i}
            features.append(shapely_to_json_feature(g, props))

        # Add point features for every single borehole intersection
        for i, pt_geom in enumerate(borehole_point_geoms):
            props = {"dataset_id": dataset_id, "kind": "borehole_intersection", "index": i}
            features.append(shapely_to_json_feature(pt_geom, props))

        out_filename = os.path.join(mesh_output_dir, f"dataset_{dataset_id:04d}.json")
        save_json_features(features, orientations, out_filename)

        if dataset_id % 10 == 0:
            print(f"Saved dataset {dataset_id} -> {out_filename}")

    print("Generation complete. Datasets saved in:", mesh_output_dir)



def visualize_dataset_pyvista(dataset_json_path, surface_path=None):
    import pyvista as pv
    import numpy as np
    import json
    import os

    with open(dataset_json_path, "r", encoding="utf-8") as f:
        fc = json.load(f)

    slice_lines, bore_lines = [], []
    orientations = fc.get("orientations", [])

    def to3(pt):
        return (float(pt[0]), float(pt[1]), float(pt[2]))

    # Build PyVista line segments for slices & boreholes
    for feat in fc.get("features", []):
        geom = feat.get("geometry")
        if not geom:
            continue
        gtype = geom.get("type")
        coords = geom.get("coordinates")
        if not coords:
            continue
        kind = feat.get("properties", {}).get("kind", "")
        if gtype == "LineString":
            pts = [to3(c) for c in coords]
            if len(pts) < 2:
                continue
            line = pv.Spline(pts, n_points=len(pts))
            (slice_lines if kind == "slice" else bore_lines).append(line)
        elif gtype == "MultiLineString":
            for part in coords:
                pts = [to3(c) for c in part]
                if len(pts) < 2:
                    continue
                line = pv.Spline(pts, n_points=len(pts))
                (slice_lines if kind == "slice" else bore_lines).append(line)

    ori_arr = np.array(orientations) if orientations else np.empty((0, 6))
    ori_points, ori_dirs = None, None
    if ori_arr.size:
        ori_points = ori_arr[:, :3]
        ori_dirs = ori_arr[:, 3:6]

    plotter = pv.Plotter()
    plotter.set_background("white")

    # Optional: show model surface for context
    if surface_path and os.path.exists(surface_path):
        try:
            surf = pv.read(surface_path)
            surf = surf.extract_surface()
            plotter.add_mesh(surf, color="lightgray", opacity=0.2, label="Surface")
        except Exception as e:
            print("Could not load surface:", e)

    # Add slices (red)
    for ln in slice_lines:
        plotter.add_mesh(ln, color="red", line_width=2, label="Slices")

    # Add boreholes (blue)
    for ln in bore_lines:
        plotter.add_mesh(ln, color="blue", line_width=3, label="Boreholes")

    # Add orientation arrows (black) — loop for each vector
    if ori_points is not None and ori_points.size:
        for p, d in zip(ori_points, ori_dirs):
            # Normalize direction (should already be normalized, but be safe)
            d = np.array(d, dtype=float)
            nrm = np.linalg.norm(d)
            if nrm < 1e-12:
                continue
            d = d / nrm
            arrow = pv.Arrow(start=tuple(p), direction=tuple(d), scale=300)  # tweak scale to taste
            plotter.add_mesh(arrow, color="black")

    plotter.add_axes(line_width=2, labels_off=False)
    plotter.add_legend()
    plotter.show_bounds(grid="front", location="outer", all_edges=True)
    plotter.show(title=f"Interactive View: {os.path.basename(dataset_json_path)}")


# === Main ===
if __name__ == "__main__":
    generate_random_datasets(input_vtk_path, output_base_dir, N_DATASETS, SLICES_PER_DATASET, BOREHOLES_PER_DATASET)
    chosen = input("Enter dataset ID to visualize (0..999), or blank to exit: ").strip()
    if chosen:
        idx = int(chosen)
        mesh_name = os.path.splitext(os.path.basename(input_vtk_path))[0]
        fname = os.path.join(output_base_dir, mesh_name, f"dataset_{idx:04d}.json")
        visualize_dataset_pyvista(fname, surface_path=input_vtk_path)

In [ ]:
"""convert all created syntehetic datasets from .json to separate csv files of orientation and interface data"""
import os
import json
import math
import re
import numpy as np
import pandas as pd
from shapely.geometry import shape, LineString, MultiLineString


# === ROOT DIRECTORY ===
ROOT_DIR = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal"


def find_json_dirs(root_dir):
    return [
        current_root
        for current_root, _, _ in os.walk(root_dir)
        if os.path.basename(current_root).lower() == "json"
    ]


def ensure_3d_coord(coord):
    try:
        if coord and len(coord) >= 3:
            return float(coord[0]), float(coord[1]), float(coord[2])
        elif coord and len(coord) == 2:
            return float(coord[0]), float(coord[1]), math.nan
    except Exception:
        pass
    return math.nan, math.nan, math.nan


def dataset_id_from_filename(fname):
    m = re.search(r"dataset[_\-]?0*([0-9]+)", fname, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def json_to_two_csvs(json_path, out_points_dir, out_ori_dir):
    base = os.path.splitext(os.path.basename(json_path))[0]
    dataset_id = dataset_id_from_filename(base)

    with open(json_path, "r", encoding="utf-8") as fh:
        fc = json.load(fh)

    points_rows = []
    orientations_rows = []

    # --- POINTS ---
    for feat in fc.get("features", []):
        geom = feat.get("geometry")
        props = feat.get("properties", {}) or {}

        formation = props.get("dataset_id", dataset_id)

        if not geom:
            continue

        try:
            s = shape(geom)
        except Exception:
            coords = geom.get("coordinates")
            if geom.get("type") == "LineString":
                s = LineString(coords)
            elif geom.get("type") == "MultiLineString":
                s = [LineString(part) for part in coords]
            else:
                continue

        if isinstance(s, LineString):
            parts = [list(s.coords)]
        elif isinstance(s, MultiLineString):
            parts = [list(p.coords) for p in s.geoms]
        elif isinstance(s, list):
            parts = [list(p.coords) for p in s if isinstance(p, LineString)]
        else:
            continue

        for coords in parts:
            for pt in coords:
                x, y, z = ensure_3d_coord(pt)
                points_rows.append({
                    "x": x,
                    "y": y,
                    "z": z,
                    "formation": f"R_{formation}"
                })

    # --- ORIENTATIONS ---
    for entry in fc.get("orientations", []):
        if isinstance(entry, dict):
            x = entry.get("X") or entry.get("x")
            y = entry.get("Y") or entry.get("y")
            z = entry.get("Z") or entry.get("z")
            gx = entry.get("G_x") or entry.get("gx")
            gy = entry.get("G_y") or entry.get("gy")
            gz = entry.get("G_z") or entry.get("gz")
        elif isinstance(entry, (list, tuple, np.ndarray)):
            e = list(entry) + [None] * (6 - len(entry))
            x, y, z, gx, gy, gz = e[:6]
        else:
            continue

        try:
            x, y, z = float(x), float(y), float(z)
            gx, gy, gz = float(gx), float(gy), float(gz)
        except Exception:
            x = y = z = gx = gy = gz = math.nan

        orientations_rows.append({
            "X": x,
            "Y": y,
            "Z": z,
            "G_x": gx,
            "G_y": gy,
            "G_z": gz,
            "formation": f"R_{dataset_id}"
        })

    # --- WRITE FILES ---
    os.makedirs(out_points_dir, exist_ok=True)
    os.makedirs(out_ori_dir, exist_ok=True)

    out_points = os.path.join(out_points_dir, f"{base}_points.csv")
    out_ori = os.path.join(out_ori_dir, f"{base}_orientations.csv")

    pd.DataFrame(points_rows, columns=["x", "y", "z", "formation"]) \
        .to_csv(out_points, index=False)

    pd.DataFrame(orientations_rows, columns=["X", "Y", "Z", "G_x", "G_y", "G_z", "formation"]) \
        .to_csv(out_ori, index=False)

    print(f"Wrote: {out_points} ({len(points_rows)} rows)")
    print(f"Wrote: {out_ori} ({len(orientations_rows)} rows)")


def main():
    json_dirs = find_json_dirs(ROOT_DIR)

    if not json_dirs:
        print("No JSON directories found.")
        return

    for json_dir in json_dirs:
        parent = os.path.dirname(json_dir)

        out_points_dir = os.path.join(parent, "csv points")
        out_ori_dir = os.path.join(parent, "csv orientations")

        json_files = sorted(f for f in os.listdir(json_dir) if f.endswith(".json"))

        if not json_files:
            continue

        print(f"\nProcessing: {json_dir}")

        for fn in json_files:
            json_path = os.path.join(json_dir, fn)
            try:
                json_to_two_csvs(json_path, out_points_dir, out_ori_dir)
            except Exception as e:
                print(f"Error in {fn}: {e}")


if __name__ == "__main__":
    main()

In [ ]:
"""1. Interpolation of datasets with GemPy,2. conversion of realizations into grid representation, 3. computation of block matrices 
(to check in which cells a realization is present --> 2 = inside, 1 = outside), and 4. summation of cell values to obtain threshold models
5. Deletion of internal cellfaces"""

import os
import json
import numpy as np
import pyvista as pv
import gempy as gp
from tqdm import tqdm
from gempy_engine.core.data.kernel_classes.kernel_functions import AvailableKernelFunctions
from gempy.modules.json_io.json_operations import JsonIO
from gempy_engine.config import AvailableBackends

# === TOP-LEVEL BASE DIRECTORY ===
root_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal"

# --- load reference mesh and compute extents ---
input_vtk_path = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Juliusburg_Koestorf-Rosenthal.vtk"
mesh = pv.read(input_vtk_path)
bounds = mesh.bounds
regular_box = pv.Box(bounds)
box_vertices = regular_box.points

Xmin, Xmax = np.min(box_vertices[:, 0]), np.max(box_vertices[:, 0])
Ymin, Ymax = np.min(box_vertices[:, 1]), np.max(box_vertices[:, 1])
Zmin, Zmax = np.min(box_vertices[:, 2]), np.max(box_vertices[:, 2])

THRESHOLDS = [199, 175, 150, 125, 101]

# ============================================================
# MAIN LOOP — ONE base_dir AT A TIME
# ============================================================
for base_dir in [
    os.path.join(root_dir, d)
    for d in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, d))
]:
    print("\n===================================================")
    print("Processing folder:", base_dir)
    print("===================================================")

    try:
        # ------------------------------------------------
        # Directory setup
        # ------------------------------------------------
        points_dir = os.path.join(base_dir, "csv points")
        orientations_dir = os.path.join(base_dir, "csv orientations")

        vtk_real_dir = os.path.join(base_dir, "vtk realizations")
        json_real_dir = os.path.join(base_dir, "json realizations")
        block_dir = os.path.join(base_dir, "block_matrices")

        vtk_blocks_dir = os.path.join(base_dir, "vtk blocks")
        vtk_hulls_dir = os.path.join(base_dir, "vtk hulls")
        json_blocks_dir = os.path.join(base_dir, "json blocks")

        for d in [
            vtk_real_dir, json_real_dir, block_dir,
            vtk_blocks_dir, vtk_hulls_dir, json_blocks_dir
        ]:
            os.makedirs(d, exist_ok=True)

        points_files = sorted(f for f in os.listdir(points_dir) if f.endswith(".csv"))
        ori_files = sorted(f for f in os.listdir(orientations_dir) if f.endswith(".csv"))

        # ------------------------------------------------
        # 1) GEMPY REALIZATIONS + BLOCK MATRICES
        # -----------------------------------------------
        START_INDEX = 0


        
        for points_file, ori_file in zip(points_files[START_INDEX:], ori_files[START_INDEX:]):
            try:
                dataset_id = os.path.splitext(points_file)[0].split("_")[1]
                project_name = f"{os.path.basename(base_dir)}_{dataset_id}"

                geo_model = gp.create_geomodel(
                    project_name=project_name,
                    extent=[Xmin, Xmax, Ymin, Ymax, Zmin, Zmax],
                    resolution=[50, 50, 50],
                    importer_helper=gp.data.ImporterHelper(
                        path_to_surface_points=os.path.join(points_dir, points_file),
                        path_to_orientations=os.path.join(orientations_dir, ori_file),
                    )
                )

                opts = geo_model.interpolation_options.kernel_options
                opts.uni_degree = 2
                opts.kernel_function = AvailableKernelFunctions.cubic

                s = gp.compute_model(
                    geo_model,
                    engine_config=gp.data.GemPyEngineConfig(
                        backend=AvailableBackends.PYTORCH
                    )
                )

                # --- save realization VTK ---
                try:
                    import gempy_viewer as gpv
                    p = gpv.plot_3d(
                        model=geo_model,
                        show_surfaces=False,
                        show_data=True,
                        show_lith=False,
                        image=True
                    )
                    surf_key = list(p.surface_poly.keys())[0]
                    p.surface_poly[surf_key].save(
                        os.path.join(vtk_real_dir, f"{project_name}.vtk")
                    )
                except Exception:
                    pass

                # --- save realization JSON ---
                try:
                    JsonIO.save_model_to_json(
                        geo_model,
                        os.path.join(json_real_dir, f"{project_name}.json")
                    )
                except Exception:
                    with open(
                        os.path.join(json_real_dir, f"{project_name}.json"), "w"
                    ) as f:
                        json.dump(geo_model.to_dict(), f, indent=2)

                # --- save block matrix ---
                #block_raw = np.array(s.raw_arrays.block_matrix)
                block_raw = np.array(s.raw_arrays.block_matrix, copy=True)
                block_raw = np.ascontiguousarray(s.raw_arrays.block_matrix)
                block_matrix = block_raw[:50*50*50].reshape((50, 50, 50))

              #  flat = block_raw.ravel()
               # flat = flat[:50 * 50 * 50]
                #block_matrix = flat.reshape((50, 50, 50))

                np.save(
                    os.path.join(
                        block_dir,
                        f"block_matrix_{os.path.basename(base_dir)}_{dataset_id}.npy"
                    ),
                    block_matrix
                )

            except Exception as e:
                print("Dataset failed:", e)
                continue

        # ------------------------------------------------
        # 2) COMBINE BLOCK MATRICES
        # ------------------------------------------------
        block_files = [
            os.path.join(block_dir, f)
            for f in os.listdir(block_dir)
            if f.endswith(".npy")
        ]

        if not block_files:
            print("No block matrices found — skipping folder.")
            continue

        combined = np.zeros_like(np.load(block_files[0]), dtype=np.int32)
        for bf in block_files:
            combined += np.load(bf).astype(np.int32)

        np.save(
            os.path.join(base_dir, f"{os.path.basename(base_dir)}_combined_block_matrix.npy"),
            combined
        )

        nx, ny, nz = combined.shape
        spacing = (
            (Xmax - Xmin) / nx,
            (Ymax - Ymin) / ny,
            (Zmax - Zmin) / nz,
        )
        origin = (Xmin, Ymin, Zmin)

        # ------------------------------------------------
        # 3) BLOCK MODELS (CUBES)
        # ------------------------------------------------
        for thr in THRESHOLDS:
            mask = combined >= thr
            if not mask.any():
                continue

            mb = pv.MultiBlock()
            for i, j, k in tqdm(np.column_stack(np.nonzero(mask)),
                                desc=f"Blocks thr {thr}"):
                center = (
                    origin[0] + (i + 0.5) * spacing[0],
                    origin[1] + (j + 0.5) * spacing[1],
                    origin[2] + (k + 0.5) * spacing[2],
                )
                mb.append(pv.Cube(center=center,
                                  x_length=spacing[0],
                                  y_length=spacing[1],
                                  z_length=spacing[2]))

            cubes = mb.combine()
            cubes.save(
                os.path.join(
                    vtk_blocks_dir,
                    f"{os.path.basename(base_dir)}_block_cubes_GE_{thr}.vtk"
                )
            )

        # ------------------------------------------------
        # 4) HULL MODELS
        # ------------------------------------------------
        for thr in THRESHOLDS:
            mask = combined >= thr
            if not mask.any():
                continue

            dims = (nx + 1, ny + 1, nz + 1)
            grid = pv.ImageData(dimensions=dims, spacing=spacing, origin=origin)
            grid.cell_data["mask"] = mask.astype(np.uint8).ravel(order="F")

            masked = grid.threshold(0.5, scalars="mask")
            hull = masked.extract_surface().clean()

            hull.save(
                os.path.join(
                    vtk_hulls_dir,
                    f"{os.path.basename(base_dir)}_hull_GE_{thr}.vtk"
                )
            )

        print("Finished folder:", base_dir)

    except Exception as e:
        print("Folder failed:", e)
        continue

print("\nALL FOLDERS PROCESSED.")


In [ ]:
"""
Smooth the surface of the created threshold models to allow for the computation of meaningful gradient data
"""

import os
import vtk

# ----------------- USER PARAMETERS -----------------
root_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal"

iterations = 10          # typical 10-100
pass_band = 0.001        # smaller = smoother
boundary_smoothing = True
feature_edge_smoothing = False
feature_angle = 120.0
# ---------------------------------------------------

def read_vtk_any(filename):
    """Read a legacy .vtk (polydata or unstructured). Returns vtkDataSet."""
    reader = vtk.vtkGenericDataObjectReader()
    reader.SetFileName(filename)
    reader.Update()
    data = reader.GetOutput()
    if data is None:
        raise RuntimeError(f"Failed to read {filename}")
    return data

def dataset_to_polydata(dataset):
    """If dataset is polydata, return it. Otherwise extract surface."""
    if dataset.IsA("vtkPolyData"):
        return dataset
    surf = vtk.vtkDataSetSurfaceFilter()
    surf.SetInputData(dataset)
    surf.Update()
    return surf.GetOutput()

def smooth_polydata(poly, iterations=30, pass_band=0.001,
                    boundary_smoothing=True,
                    feature_edge_smoothing=False,
                    feature_angle=120.0):
    """Smooth polydata using vtkWindowedSincPolyDataFilter and recompute normals."""
    smoother = vtk.vtkWindowedSincPolyDataFilter()
    smoother.SetInputData(poly)
    smoother.SetNumberOfIterations(int(iterations))
    smoother.SetPassBand(float(pass_band))
    smoother.SetFeatureEdgeSmoothing(bool(feature_edge_smoothing))
    smoother.SetBoundarySmoothing(bool(boundary_smoothing))
    smoother.SetFeatureAngle(float(feature_angle))
    smoother.NonManifoldSmoothingOn()
    smoother.NormalizeCoordinatesOn()
    smoother.Update()

    normals = vtk.vtkPolyDataNormals()
    normals.SetInputConnection(smoother.GetOutputPort())
    normals.ComputePointNormalsOn()
    normals.ComputeCellNormalsOff()
    normals.AutoOrientNormalsOn()
    normals.Update()

    return normals.GetOutput()

def write_polydata_vtk(poly, filename):
    writer = vtk.vtkPolyDataWriter()
    writer.SetFileName(filename)
    if vtk.VTK_MAJOR_VERSION <= 5:
        writer.SetInput(poly)
    else:
        writer.SetInputData(poly)
    writer.Write()
    print(f"Wrote VTK: {filename}")

def process_vtk_file(vtk_file):
    print("Processing:", vtk_file)
    data = read_vtk_any(vtk_file)
    poly = dataset_to_polydata(data)
    smoothed = smooth_polydata(poly,
                               iterations=iterations,
                               pass_band=pass_band,
                               boundary_smoothing=boundary_smoothing,
                               feature_edge_smoothing=feature_edge_smoothing,
                               feature_angle=feature_angle)
    output_file = os.path.splitext(vtk_file)[0] + "_smoothed.vtk"
    write_polydata_vtk(smoothed, output_file)

def main():
    # Iterate through all directories matching the "x sections y boreholes" pattern
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if os.path.basename(dirpath) == "vtk hulls":
            print(f"Found 'vtk hulls' folder: {dirpath}")
            for f in filenames:
                if f.endswith(".vtk") and "_block_cubes_" not in f:
                    vtk_file = os.path.join(dirpath, f)
                    process_vtk_file(vtk_file)
    print("All done.")

if __name__ == "__main__":
    main()
